In [1]:
%pip install openpyxl 
%pip install pandas 
%pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import ipywidgets as widgets
import io
from pathlib import Path
from IPython.display import display

In [ ]:
def baca_file(uploader, label):
    if not uploader.value:
        raise ValueError(f"File {label} belum diupload!")
    item = uploader.value[0]
    nama = item['name']
    buf  = io.BytesIO(item['content'])
    ext  = Path(nama).suffix.lower()

    if ext == ".xlsx":
        df = pd.read_excel(buf, engine="openpyxl")
    elif ext == ".xls":
        df = pd.read_excel(buf, engine="xlrd")
    elif ext in (".csv", ".tsv"):
        sep = "\t" if ext == ".tsv" else ","
        for enc in ("utf-8", "utf-8-sig", "latin-1"):
            try:
                buf.seek(0)
                df = pd.read_csv(buf, sep=sep, encoding=enc)
                break
            except UnicodeDecodeError:
                continue
        else:
            raise ValueError(f"Tidak dapat membaca: {nama}")
    else:
        raise ValueError(f"Format tidak didukung: '{ext}'")

    print(f"✅ '{nama}' — {len(df):,} baris, {len(df.columns)} kolom")
    return df


def validasi_kolom(df, nama):
    kurang = {"sent_id", "misc", "tipe_afiks"} - set(df.columns)
    if kurang:
        raise ValueError(f"File '{nama}' kekurangan kolom: {kurang}\nKolom ada: {list(df.columns)}")

In [6]:
uploader_patokan = widgets.FileUpload(
    accept='.xlsx,.xls,.csv,.tsv', multiple=False, description='📂 Patokan')
uploader_cek = widgets.FileUpload(
    accept='.xlsx,.xls,.csv,.tsv', multiple=False, description='📂 Dicek')

display(widgets.VBox([
    widgets.Label("File patokan (ground truth):"),
    uploader_patokan,
    widgets.Label("File yang dicek:"),
    uploader_cek,
]))

In [16]:
print("Membaca file patokan...")
df_patokan = baca_file(uploader_patokan, "patokan")
validasi_kolom(df_patokan, "patokan")

print("\nMembaca file yang dicek...")
df_cek = baca_file(uploader_cek, "dicek")
validasi_kolom(df_cek, "dicek")

# Normalisasi semua kolom kunci
for df in [df_patokan, df_cek]:
    df["sent_id"]    = df["sent_id"].astype(str).str.strip()
    df["misc"]       = df["misc"].astype(str).str.strip()
    df["tipe_afiks"] = df["tipe_afiks"].astype(str).str.strip()

print("\n── Preview patokan ──")
display(df_patokan[["sent_id", "token", "misc", "tipe_afiks"]].head())
print("── Preview file dicek ──")
display(df_cek[["sent_id", "token", "misc", "tipe_afiks"]].head())

Membaca file patokan...
✅ 'ud-indo-gsd_test_morph.csv' — 10,056 baris, 15 kolom

Membaca file yang dicek...
✅ 'pemeriksaan_morfologiAfiksasi_sample_test.xlsx' — 385 baris, 12 kolom

── Preview patokan ──


,sent_id,token,misc,tipe_afiks
0,test-s1,Sampul,{'MorphInd': '^sampul<n>_NSD$'},Tidak Berimbuhan
1,test-s1,dari,{'MorphInd': '^dari<r>_R--$'},Tidak Berimbuhan
2,test-s1,dua,{'MorphInd': '^dua<c>_CC-$'},Tidak Berimbuhan
3,test-s1,singel,{'MorphInd': '^singel<x>_X--$'},Tidak Berimbuhan
4,test-s1,pertama,{'MorphInd': '^pertama<c>_CO-$'},Tidak Berimbuhan


── Preview file dicek ──


,sent_id,token,misc,tipe_afiks
0,test-s465,tegas,{'MorphInd': '^tegas<a>_ASP$'},Tidak Berimbuhan
1,test-s304,dari,{'MorphInd': '^dari<r>_R--$'},Tidak Berimbuhan
2,test-s62,muda,{'MorphInd': '^muda<a>_ASP$'},Tidak Berimbuhan
3,test-s484,panen,{'MorphInd': '^panen<n>_NSD$'},Tidak Berimbuhan
4,test-s6,merupakan,{'MorphInd': '^merupakan<o>_O--$'},Tidak Berimbuhan


In [17]:
hasil_beda    = []
tidak_ketemu  = []

for _, baris in df_patokan.iterrows():
    sid       = baris["sent_id"]
    misc      = baris["misc"]
    tipe_pat  = baris["tipe_afiks"]
    token     = str(baris.get("token", "")).strip()

    # Cari baris di file cek: sent_id sama LALU misc sama
    baris_cek = df_cek[
        (df_cek["sent_id"] == sid) &
        (df_cek["misc"]    == misc)
    ]

    if baris_cek.empty:
        tidak_ketemu.append({
            "sent_id" : sid,
            "token"   : token,
            "misc"    : misc,
            "tipe_patokan": tipe_pat,
        })
    else:
        tipe_cek = baris_cek.iloc[0]["tipe_afiks"]
        if tipe_pat != tipe_cek:
            hasil_beda.append({
                "sent_id"     : sid,
                "token"       : token,
                "misc"        : misc,
                "tipe_patokan": tipe_pat,
                "tipe_cek"    : tipe_cek,
            })

total   = len(df_patokan)
beda    = len(hasil_beda)
tdk_ada = len(tidak_ketemu)
cocok   = total - beda - tdk_ada
akurasi = cocok / total * 100 if total > 0 else 0

print("=" * 50)
print(f"  Baris patokan   : {total:>6,}")
print(f"  Cocok           : {cocok:>6,}")
print(f"  Berbeda         : {beda:>6,}")
print(f"  Tidak ditemukan : {tdk_ada:>6,}")
print(f"\n  🎯 Akurasi      : {akurasi:>6.2f}%")
print("=" * 50)

  Baris patokan   : 10,056
  Cocok           :    458
  Berbeda         :     11
  Tidak ditemukan :  9,587

  🎯 Akurasi      :   4.55%


In [18]:
if beda == 0 and tdk_ada == 0:
    print("✅ Tidak ada perbedaan! Semua anotasi sesuai patokan.")

if beda > 0:
    print(f"⚠️  {beda} baris BERBEDA:")
    print(f"{'='*70}")
    print(f"{'No':<5} {'sent_id':<18} {'token':<18} {'patokan':<22} {'cek'}")
    print(f"{'-'*5} {'-'*18} {'-'*18} {'-'*22} {'-'*18}")
    for i, row in enumerate(hasil_beda):
        print(f"{i+1:<5} {row['sent_id']:<18} {row['token']:<18} "
              f"{row['tipe_patokan']:<22} {row['tipe_cek']}")
    print(f"{'='*70}")

if tdk_ada > 0:
    print(f"\n📌 {tdk_ada} baris TIDAK DITEMUKAN di file cek:")
    print(f"{'='*70}")
    print(f"{'No':<5} {'sent_id':<18} {'token':<18} {'misc'}")
    print(f"{'-'*5} {'-'*18} {'-'*18} {'-'*40}")
    for i, row in enumerate(tidak_ketemu):
        print(f"{i+1:<5} {row['sent_id']:<18} {row['token']:<18} {row['misc']}")
    print(f"{'='*70}")

⚠️  11 baris BERBEDA:
No    sent_id            token              patokan                cek
----- ------------------ ------------------ ---------------------- ------------------
1     test-s113          tersebut           Tidak Berimbuhan       Prefiks
2     test-s149          diseluruh          Sufiks                 Tidak Berimbuhan
3     test-s225          berhasil           Tidak Berimbuhan       Prefiks
4     test-s235          umumnya            Tidak Berimbuhan       Sufiks
5     test-s341          berbagai           Tidak Berimbuhan       Prefiks
6     test-s355          biasanya           Tidak Berimbuhan       Sufiks
7     test-s369          gundukan           Tidak Berimbuhan       Sufiks
8     test-s413          Kuningan           Sufiks                 Tidak Berimbuhan
9     test-s428          umumnya            Tidak Berimbuhan       Sufiks
10    test-s478          berbagai           Tidak Berimbuhan       Prefiks
11    test-s515          selanjutnya        Tidak Berimbu